In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 19.1 MB/s eta 0:00:00


In [ ]:
import optuna
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import numpy as np


In [ ]:
(x_train,y_train),(x_test,y_test) = keras.datasets.mnist.load_data()
x = np.concatenate((x_train,x_test), axis = 0 ).astype("float32")/255.0
y = np.concatenate((y_train,y_test), axis = 0)

x = x.reshape((-1,28*28))
x_train,x_val,y_train,y_val = train_test_split(x,y,test_size = 0.2,random_state=42)


print(x_train.shape)
print(x_val.shape)
print(y_train.shape)
print(y_val.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
(56000, 784)
(14000, 784)
(56000,)
(14000,)


In [ ]:
import optuna
from tensorflow import keras
from tensorflow.keras import layers


def objective(trial):

    # Hyperparameters
    hidden_units = trial.suggest_int("hidden_units", 64, 512)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    learning_rate = trial.suggest_float(
        "learning_rate", 1e-5, 1e-2, log=True
    )
    optimizer_name = trial.suggest_categorical(
        "optimizer", ["adam", "sgd"]
    )

    # Model definition
    model = keras.Sequential([
        keras.Input(shape=(28 * 28,)),
        layers.Dense(hidden_units, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation="softmax")
    ])

    # Optimizer
    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(
            learning_rate=learning_rate
        )
    else:
        optimizer = keras.optimizers.SGD(
            learning_rate=learning_rate
        )

    # Compile model
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Training
    history = model.fit(
        x_train,
        y_train,
        batch_size=128,
        epochs=10,
        validation_data=(x_val, y_val),
        verbose=0
    )

    # Return best validation accuracy
    return max(history.history["val_accuracy"])




In [ ]:
# Create Optuna study
study = optuna.create_study(direction="maximize")

# Run optimization
study.optimize(objective, n_trials=50)


# Display best trial
print("Best Trial:")

trial = study.best_trial

print(f"Accuracy: {trial.value}")

print("Params:")

for key, value in trial.params.items():
    print(f"{key}: {value}")

[I 2026-10-07 06:38:17,609] A new study created in memory with name: no-name-845b2143-cc83-4316-90a1-5fa3d9273d78
[I 2026-10-07 06:38:59,401] Trial 0 finished with value: 0.8442142605781555 and parameters: {'hidden_units': 267, 'dropout_rate': 0.19750627132879864, 'learning_rate': 0.0007924249711763985, 'optimizer': 'sgd'}. Best is trial 0 with value: 0.8442142605781555.
[I 2026-10-07 06:39:40,614] Trial 1 finished with value: 0.972000002861023 and parameters: {'hidden_units': 189, 'dropout_rate': 0.13695556362502087, 'learning_rate': 0.007747066181935687, 'optimizer': 'adam'}. Best is trial 1 with value: 0.972000002861023.
[I 2026-10-07 06:40:20,414] Trial 2 finished with value: 0.9269999861717224 and parameters: {'hidden_units': 198, 'dropout_rate': 0.2818808228943615, 'learning_rate': 3.654070625921323e-05, 'optimizer': 'adam'}. Best is trial 1 with value: 0.972000002861023.
[I 2026-10-07 06:40:55,211] Trial 3 finished with value: 0.6860714554786682 and parameters: {'hidden_units': 

Best Trial:
Accuracy: 0.9809285998344421
Params:
hidden_units: 420
dropout_rate: 0.29361759241932156
learning_rate: 0.0013524135439384214
optimizer: adam


In [ ]:
optuna.visualization.plot_param_importances(study).show()

In [ ]:
def objective(trail):
    # Hyperparameters
    n_layers = trail.suggest_int("n_layers",1,3)
    hidden_units = trail.suggest_int("hidden_units",64,512)
    droupout_rate = trail.suggest_float("droupout_rate",0.1,0.5)
    learning_rate = trail.suggest_float("learning_rate",1e-5,1e-2,log = True)
    optimizer = trail.suggest_categorical("optimizer",["adam","sgd"])

    # Model definition
    model = keras.Sequential()
    model.add(layers.Input(shape = (28*28,)))

    for i in range(n_layers):
        model.add(layers.Dense(hidden_units,activation = "relu",name = f'hidden_{i+1}'))
        model.add(layers.Dropout(droupout_rate,name = f'hidden_{i+1}'))
    model.add(layers.Dense(10,activation = "softmax",name= 'output'))

    # Compile model
    optimizer = getattr(keras.optimizers,optimizer)(learning_rate = learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    # Training
    history = model.fit(
        x_train,
        y_train,
        batch_size=128,
        epochs=10,
        validation_data=(x_val, y_val),
        verbose=0
    )

    # Return best validation accuracy
    return max(history.history["val_accuracy"])
